# MLflow — Gestión de Experimentos con AutoML Paralelo

**MCD-402 · MLOps · 2026-2**

Este notebook muestra el flujo completo de un experimento reproducible:
cómo registrar cada entrenamiento en MLflow, comparar resultados y cargar
el modelo campeón — con paralelismo CPU y soporte GPU integrado en el pipeline.

---
**Flujo:**
```
Cargar datos → Entrenar (paralelo + GPU si disponible) → Registrar en MLflow → Comparar → Cargar campeón
```

## 1. Setup — Importaciones y configuración

In [ ]:
import os, sys
import yaml
import pandas as pd
import numpy as np
import mlflow
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Agregar la raíz del proyecto al path para importar src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.training  import run_parallel_hpo, detect_gpu
from src.registry  import setup_mlflow, log_all_runs, get_best_run, load_champion

# Leer configuración del proyecto
with open(os.path.join(PROJECT_ROOT, 'config', 'pipeline_config.yml')) as f:
    CFG = yaml.safe_load(f)

MLFLOW_DB   = os.path.join(PROJECT_ROOT, 'mlflow', 'mlflow.db')
TRACKING    = f'sqlite:///{MLFLOW_DB}'
ARTIFACT_RT = os.path.join(PROJECT_ROOT, CFG['mlflow']['artifact_root'])
EXP_NAME    = CFG['mlflow']['experiment_name']

gpu = detect_gpu()
print(f'Proyecto: {PROJECT_ROOT}')
print(f'GPU disponible: {gpu}')
print(f'Experimento MLflow: {EXP_NAME}')

## 2. Preparar datos

Si tienes datos reales en `data/processed/train.csv` y `test.csv`, cárgalos con `pd.read_csv`.
Si no, el bloque siguiente genera un dataset sintético equivalente al caso de uso del curso.

In [ ]:
TRAIN_PATH = os.path.join(PROJECT_ROOT, CFG['data']['train_path'])
TEST_PATH  = os.path.join(PROJECT_ROOT, CFG['data']['test_path'])

if os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH):
    train_df = pd.read_csv(TRAIN_PATH)
    test_df  = pd.read_csv(TEST_PATH)
    print('Datos reales cargados.')
else:
    print('Datos reales no encontrados — generando dataset sintético...')
    X, y = make_classification(
        n_samples=30_000, n_features=25, n_informative=15,
        random_state=42, class_sep=0.8
    )
    df = pd.DataFrame(X, columns=[f'v{i}' for i in range(X.shape[1])])
    df.insert(0, 'target', y)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
    os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)
    train_df.to_csv(TRAIN_PATH, index=False)
    test_df.to_csv(TEST_PATH,   index=False)

TARGET  = train_df.columns[0]
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=[TARGET]).reindex(columns=X_train.columns, fill_value=0)
y_test  = test_df[TARGET]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Balance target — train: {y_train.mean():.2%}  |  test: {y_test.mean():.2%}')

## 3. Iniciar MLflow

El servidor se conecta a una base SQLite local. No necesitas Docker ni conexión a internet.

> **¿Qué hace MLflow?**  
> Registra cada ejecución de entrenamiento (*run*) con sus parámetros, métricas y artefactos
> para que puedas comparar experimentos y reproducir cualquier resultado.

```
mlflow/
├── mlflow.db              ← base SQLite de runs
└── artifacts/
    └── AutoML_Pipeline/
        ├── run_001/
        │   ├── params/     ← hiperparámetros
        │   ├── metrics/    ← auc, decay, speedup
        │   └── graficos/   ← curva ROC
        └── run_002/
```

In [ ]:
os.makedirs(os.path.dirname(MLFLOW_DB), exist_ok=True)
os.makedirs(ARTIFACT_RT, exist_ok=True)

setup_mlflow(TRACKING, ARTIFACT_RT, EXP_NAME)

print(f'Tracking URI: {TRACKING}')
print(f'Experimento:  {EXP_NAME}')

## 4. Entrenar y registrar en MLflow

El pipeline:
1. **HPO** con `RandomizedSearchCV` sobre XGBoost, LightGBM y CatBoost
2. **Paralelismo**: primero secuencial (para medir tiempo base), luego `joblib.Parallel` con backend `loky`
3. **GPU**: si hay CUDA disponible, se activa automáticamente en XGBoost, LGBM y CatBoost
4. **MLflow**: cada modelo queda registrado como un *run* independiente

> No hay código separado de "demo de paralelismo" — el speedup y el uso de GPU son
> métricas que MLflow registra junto con el AUC de cada modelo.

In [ ]:
results, champion, par_meta = run_parallel_hpo(
    X_train, y_train, X_test, y_test,
    n_iter  = CFG['training']['n_iter_hpo'],
    cv      = CFG['training']['cv_folds'],
    n_jobs  = CFG['training']['n_jobs'],
    random_state    = CFG['training']['random_state'],
    decay_threshold = CFG['training']['decay_threshold'],
    use_gpu         = CFG['training']['use_gpu'],
)

print(f"\nSpeedup CPU: {par_meta['speedup']}×  "
      f"({par_meta['t_sequential_s']}s → {par_meta['t_parallel_s']}s)")
print(f"GPU usada:   {par_meta['gpu_used']}")
if champion:
    print(f"\nCampeón:  {champion['name']}  AUC={champion['auc_test']}  decay={champion['decay_pct']}%")

In [ ]:
# Registrar todos los runs en MLflow
run_ids = log_all_runs(results, par_meta, X_test, y_test, extra_tags=CFG['mlflow']['tags'])
print(f'\n{len(run_ids)} runs registrados en experimento "{EXP_NAME}"')

## 5. Comparar experimentos con la API de MLflow

Podemos consultar los resultados directamente desde Python sin abrir la UI.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp    = client.get_experiment_by_name(EXP_NAME)
runs   = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=['metrics.auc_test DESC']
)

print(f'{'Modelo':<10} {'AUC Test':>10} {'AUC CV':>8} {'Decay':>7} {'Speedup':>9} {'GPU':>5}')
print('-' * 55)
for r in runs:
    m = r.data.metrics
    print(
        f"{r.info.run_name:<10} "
        f"{m.get('auc_test',0):>10.4f} "
        f"{m.get('auc_cv',0):>8.4f} "
        f"{m.get('decay_pct',0):>6.1f}% "
        f"{m.get('speedup_cpu',0):>8.2f}x "
        f"{r.data.tags.get('gpu_used','?'):>5}"
    )

## 6. Cargar el modelo campeón y predecir

In [ ]:
best_run = get_best_run(EXP_NAME, metric='auc_test')

modelo_cargado = load_champion(best_run.info.run_id)

muestra = X_test.iloc[:8]
proba   = modelo_cargado.predict_proba(muestra)[:, 1]
pred    = modelo_cargado.predict(muestra)

print(f'Campeón cargado: {best_run.info.run_name}')
print(f'AUC en test:     {best_run.data.metrics["auc_test"]}')
print(f'Predicciones (8 registros):')
for i, (p, pr) in enumerate(zip(pred, proba)):
    print(f'  registro {i}: clase={p}  prob_positiva={pr:.4f}')

## 7. Abrir la UI de MLflow

Ejecuta la celda siguiente para iniciar el servidor MLflow localmente.
Luego abre **http://localhost:5000** en el navegador.

> Para detenerlo: `mlflow_proc.terminate()` o cierra la terminal.

In [ ]:
import subprocess, time, socket

MLFLOW_PORT = 5000

def _port_libre(port):
    with socket.socket() as s:
        return s.connect_ex(('localhost', port)) != 0

if _port_libre(MLFLOW_PORT):
    mlflow_proc = subprocess.Popen([
        'mlflow', 'ui',
        '--backend-store-uri', TRACKING,
        '--default-artifact-root', ARTIFACT_RT,
        '--host', '0.0.0.0',
        '--port', str(MLFLOW_PORT),
    ])
    time.sleep(3)
    print(f'MLflow UI iniciado → http://localhost:{MLFLOW_PORT}')
    print(f'Para detenerlo: mlflow_proc.terminate()')
else:
    print(f'Puerto {MLFLOW_PORT} ya en uso — MLflow probablemente ya está corriendo.')
    print(f'Abre http://localhost:{MLFLOW_PORT}')

---

## Resumen

| Concepto | Qué hace | Dónde está |
|---|---|---|
| `mlflow.start_run()` | Abre un experimento y registra todo lo que pase dentro | `src/registry.py` |
| `log_params / log_metrics` | Guarda hiperparámetros y métricas asociados al run | `src/registry.py` |
| `mlflow.sklearn.log_model` | Serializa y versiona el modelo entrenado | `src/registry.py` |
| `Parallel(n_jobs=-1)` | Entrena todos los modelos en paralelo (loky) | `src/training.py` |
| `detect_gpu()` | Detecta CUDA y activa GPU en XGB/LGBM/CatBoost | `src/training.py` |
| `mlflow ui` | Abre el frontend para comparar visualmente los runs | Celda 7 |
| `dags/training_dag.py` | Orquesta todo lo anterior con Airflow + cron | `dags/` |

El cron del pipeline se configura en **`config/pipeline_config.yml`** → `airflow.cron_schedule`  
(por defecto: `0 */2 * * *` = cada 2 horas).